In [ ]:
%pip install -U accelerate peft trl transformers datasets
%pip install -U langchain-community langchain-google-genai chromadb

In [ ]:
from huggingface_hub import login
import os

hf_token = os.environ.get("HF_TOKEN")
if hf_token:  # 토큰이 있을 때만 로그인 (Qwen2.5-1.5B는 gated 모델이 아니라 없어도 실행됨)
    login(hf_token)

### 학습 진행

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# -----------------------------
# 1) 시스템 프롬프트 (학습, 추론 둘 다 이거 하나 씀)
# -----------------------------
SYSTEM_PROMPT = (
    "당신은 한국어로 답하는 어시스턴트입니다. 반드시 한국어로만 답하세요. "
    "제공된 문맥이 있으면 그 문맥을 근거로 답하세요."
)

# -----------------------------
# 2) KoAlpaca 로딩
# -----------------------------
def load_hf_dataset(repo_id="beomi/KoAlpaca-v1.1a", split="train"):
    return load_dataset(repo_id, split=split)

# -----------------------------
# 3) chat template 적용해서 text 컬럼 만들기
# -----------------------------
def build_sft_text_dataset(hf_ds, tokenizer, system_prompt):
    def _format_one(example):
        instruction = (example.get("instruction") or "").strip()
        user_input = (example.get("input") or "").strip()
        output = (example.get("output") or "").strip()

        if user_input:
            user_content = f"{instruction}\n\n[추가 입력]\n{user_input}"
        else:
            user_content = instruction

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        return {"text": text}

    return hf_ds.map(_format_one, remove_columns=hf_ds.column_names, desc="Formatting")

# -----------------------------
# 4) 모델 + LoRA
# -----------------------------
def load_model_and_tokenizer(model_id="Qwen/Qwen2.5-1.5B-Instruct"):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    device = "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"Using device: {device}")

    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32)  # MPS라 fp32
    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.to(device)
    model.print_trainable_parameters()
    return model, tokenizer

# -----------------------------
# 5) 학습
# -----------------------------
def train():
    hf_ds = load_hf_dataset("beomi/KoAlpaca-v1.1a", split="train[:1000]")
    model, tokenizer = load_model_and_tokenizer("Qwen/Qwen2.5-1.5B-Instruct")
    train_ds = build_sft_text_dataset(hf_ds, tokenizer, SYSTEM_PROMPT)

    args = SFTConfig(
        output_dir="qwen2_5_1_5b_koalpaca_lora_mps_sft_1",
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_ratio=0.1,
        learning_rate=1e-5,
        optim="adamw_torch",
        fp16=False,
        bf16=False,
        logging_steps=1,
        save_strategy="epoch",
        report_to="none",
        max_length=1024,
        packing=False,   # MPS에서 packing 켜면 샘플 섞인다는 경고 떠서 끔
        gradient_checkpointing=True,
        remove_unused_columns=False,
        shuffle_dataset=True,
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        processing_class=tokenizer,
    )
    trainer.train()

    save_dir = os.path.join(args.output_dir, "adapter")
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print("저장 완료:", save_dir)

train()

## RAG 붙여서 답변 확인

In [ ]:
import torch
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

adapter_path = "qwen2_5_1_5b_koalpaca_lora_mps_sft_1/adapter"

# 학습 셀을 건너뛰고 이 RAG 섹션만 돌릴 때도 되도록 시스템 프롬프트를 다시 정의
SYSTEM_PROMPT = (
    "당신은 한국어로 답하는 어시스턴트입니다. 반드시 한국어로만 답하세요. "
    "제공된 문맥이 있으면 그 문맥을 근거로 답하세요."
)

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
model = AutoPeftModelForCausalLM.from_pretrained(
    adapter_path,
    dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("파인튜닝 모델 로드 완료")

# API 키는 환경변수로 (export GOOGLE_API_KEY=...)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma(persist_directory="./final_legal_db", embedding_function=embeddings)

In [ ]:
def chat_with_model_rag(question, k=3):
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join(doc.page_content for doc in docs)

    print(f"\n[질문] {question}")
    print("[검색된 문서]")
    for i, doc in enumerate(docs, 1):
        print(f"  {i}. {doc.page_content[:100].strip()} ...")

    user_content = f"다음 법률 문맥을 참고하여 답변하세요:\n\n[법률 문맥]\n{context}\n\n[질문]\n{question}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
        )

    answer = tokenizer.decode(gen[0, input_len:], skip_special_tokens=True)
    print("[답변]")
    print(answer.strip())

In [ ]:
questions = [
    "신호위반 차량이 저를 들이받고 도주했습니다. 가해자가 무보험이라는데 어떻게 되나요?",
    "전세 계약이 끝났는데 집주인이 보증금을 안 돌려줘요. 어떻게 해야 하나요?",
    "회사에서 갑자기 해고 통보를 받았는데 해고예고수당을 받을 수 있나요?",
]
for q in questions:
    chat_with_model_rag(q)

- 답변에 중국어가 섞여 나올 때가 있다.
- 같은 질문을 다시 돌리면 답이 매번 다르게 나옴 (do_sample=True라 그런 듯)
- 검색된 문서랑 상관없는 답을 할 때도 있다
- 다음: 데이터셋 새로 가져와서 평가 기준 정하고 이어서 해 보기